**Monte Carlo integration of a parabola.**
The previous circle and sphere notebooks estimated geometric volumes by
counting points inside a shape.
The same idea applies to any definite integral: scatter random points over
a bounding rectangle and count how many fall under the curve.

Here we estimate $\displaystyle\int_{-2}^{2}(4-x^2)\,dx$.
The parabola $f(x) = 4 - x^2$ has roots at $x = \pm 2$ and a maximum of 4 at $x = 0$.
The exact value by the power rule is:

$$\int_{-2}^{2}(4-x^2)\,dx
= \left[4x - \frac{x^3}{3}\right]_{-2}^{2}
= \left(8 - \frac{8}{3}\right) - \left(-8 + \frac{8}{3}\right)
= 16 - \frac{16}{3} = \frac{32}{3} \approx 10.667$$


---
**Halton sequence (Numba-compiled).**
Same implementation as the sphere and hypersphere notebooks.

**Sampling region.**
The bounding rectangle spans $x \in [-2, 2]$ and $y \in [0, 5]$,
giving a sample area of $4 \times 5 = 20$.
A point $(x, y)$ is **under** the curve if $y \leq f(x)$, i.e. $y - (4-x^2) \leq 0$.

$$\hat{I} = 20 \times \frac{\text{dots under curve}}{\text{dots total}}$$

The top of the box ($y = 5$) is above the parabola's maximum ($y = 4$)
to ensure all points under the curve can be sampled.

In [ ]:
"""mc_parabola.ipynb"""

# Cell 01 - Halton quasi-random sequence, bounding rectangle, and sample count

%matplotlib inline

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import Rectangle
from numba import float64, int64, vectorize


@vectorize([float64(int64, int64)], nopython=True)
def halton(n, p):
    h, f = 0, 1
    while n > 0:
        f = f / p
        h += (n % p) * f
        n = int(n / p)
    return h


bbox = Rectangle((-2, 0), 4, 5).get_bbox()  # x in [-2,2], y in [0,5]
print(bbox)
print(f"Sample area = {bbox.width * bbox.height}")

total_dots = 40_600
print(f"{total_dots = :,}")

---
**Generating the Halton sample points.**
Base 2 for $x$, base 3 for $y$, each value in $[0,1)$ and scaled to the
bounding rectangle.

The sequence is indexed from $n = 1$: since `halton(0, p)` returns $0$
for every base, $n = 0$ would place a sample at the corner of the rectangle.

In [ ]:
# Cell 02 - Generate Halton sample points over the bounding rectangle
# Halton is indexed from n = 1; n = 0 would map to the corner of the rectangle

n_arr = np.arange(1, total_dots + 1, dtype=np.int64)

x = halton(n_arr, np.full(total_dots, 2, dtype=np.int64)) * bbox.width + bbox.x0
y = halton(n_arr, np.full(total_dots, 3, dtype=np.int64)) * bbox.height + bbox.y0

pd.DataFrame({"x": x[:5], "y": y[:5]})

---
**Signed distance from the parabola.**
$d = y - f(x) = y - (4 - x^2)$.
If $d \leq 0$ the point is on or below the curve (inside the integral region);
if $d > 0$ it is above the curve.

In [ ]:
# Cell 03 - Signed distance: d = y - f(x) = y - (4 - x^2)
# d <= 0: point is under (or on) the parabola
# d >  0: point is above the parabola

d = y - (4 - x**2)

pd.DataFrame({"x": x[:5], "y": y[:5], "d": d[:5]})

---
**Classifying points as under or above the parabola.**

In [ ]:
# Cell 04 - Partition points: under/on parabola (d <= 0) vs above (d > 0)

x_in = x[d <= 0.0]  # under or on the curve
y_in = y[d <= 0.0]
x_out = x[d > 0.0]  # above the curve
y_out = y[d > 0.0]

pd.DataFrame(
    {"x_in": x_in[:5], "y_in": y_in[:5], "x_out": x_out[:5], "y_out": y_out[:5]}
)

---
**Scatter plot with parabola overlay.**
Red points fall under or on $f(x) = 4 - x^2$; blue points are above it.
The black curve is the exact parabola, confirming the boundary between
the two populations.

In [ ]:
# Cell 05 - Scatter plot with exact parabola overlaid

x_curve = np.linspace(-2, 2, 300)
y_curve = 4 - x_curve**2

plt.figure(figsize=(7, 6))
plt.scatter(x_in, y_in, color="red", marker=".", s=0.5)
plt.scatter(x_out, y_out, color="blue", marker=".", s=0.5)
plt.plot(x_curve, y_curve, color="black", lw=1.5, label="$f(x)=4-x^2$")
plt.title("Monte Carlo Integration: $f(x) = 4 - x^2$")
plt.xlabel("x")
plt.ylabel("y")
plt.legend(loc="upper right")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
**Estimating the integral and measuring the error.**
The Monte Carlo estimate is:
$\hat{I} = A_{\text{rect}} \times (\text{dots under curve} / \text{dots total}) = 20 \times (\text{dots under} / \text{total})$.

In [ ]:
# Cell 06 - Compute the integral estimate and error

act = 32 / 3
est = (bbox.width * bbox.height) * np.count_nonzero(d <= 0) / total_dots
err = np.abs((est - act) / act)

print(f"dots total        : {total_dots:,}")
print(f"dots under curve  : {np.count_nonzero(d <= 0):,}")
print(f"actual (32/3)     : {act:.6f}")
print(f"estimated         : {est:.6f}")
print(f"abs pct rel error : {err:.5%}")